# OmniGuard: full 50-sample evaluation with three displayed examples

OmniGuard was run on **all 50 clips in the fixed MOSE selection**. To keep the notebook readable, it displays the same three fixed examples used by the other notebooks; the aggregate table below and linked evidence cover all 50.

```text
original -> framewise OmniGuard embedding -> watermarked video -> ProPainter, DiffuEraser, or EffectErase -> OmniGuard localization
```

OmniGuard is released for images. The video adapter applies it independently to each frame without resizing these 960 x 480 MOSE sources.

The released OmniGuard detector returns pixel probabilities. For binary mask display and IoU, this evaluation uses the conventional 0.5 probability threshold. OmniGuard's public demo does not expose a recovered bit message, so bit accuracy is not applicable.


## Metric guide

| Term | Meaning | Interpretation |
| --- | --- | --- |
| Embedding PSNR | Similarity between the aligned source frames and watermarked frames. | Higher is better; it measures embedding distortion, not detector quality. |
| Global pixel IoU | Total intersection divided by total union across all frames and pixels. | 1 is perfect overlap. 0 means no overlap. This is the primary localization result. |
| Mean frame IoU | IoU calculated for each frame and then averaged. | Empty ground-truth and prediction masks score 1, so global IoU remains the primary score. |
| Effective GT mask | The removal mask after the remover's actual dilation and spatial preprocessing. | This is the correct reference mask for IoU. |
| Predicted mask | The method's own detector output after documented binarization. | White indicates predicted tampering. |
| Message bit accuracy | Fraction of fixed copyright bits recovered correctly, averaged across frames. | Available for EditGuard; not exposed by OmniGuard's public demo. |
| Runtime | Wall-clock time recorded on the shared experiment hardware. | Useful for this run, not a hardware-independent benchmark. |
| SHA-256 | File fingerprint stored in the JSON record. | Confirms exact file identity. It is not a quality metric. |


## Aggregate results across all 50 selected clips

All 50 clips completed embedding, ProPainter, DiffuEraser, EffectErase, and native localization. The values below are unweighted means of the 50 per-clip results; the complete arrays are in [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json).

| Metric | ProPainter | DiffuEraser | EffectErase |
| --- | ---: | ---: | ---: |
| Mean global pixel IoU | **0.0015** | **0.0014** | **0.0025** |
| Message bit accuracy | N/A | N/A | N/A |

Mean embedding PSNR across all 50 clips: **48.8048 dB**.

The remainder of this notebook deliberately displays only three clips: `02b72b8d`, `0ab75563`, and `0b02886a`.


## Notebook setup

After a fresh clone, run `python3 scripts/prepare_mose_notebooks.py` from the repository root. This downloads and verifies the videos shared by the method notebooks. Run the next cell once to enable the video, image, download, and metadata viewers. If Git LFS has left a pointer file in place of a video, the viewer reports the preparation command instead of displaying a broken player.


In [ ]:
import base64
from pathlib import Path
import html
import json
from IPython.display import HTML, Markdown, display

LFS_POINTER_PREFIX = b"version https://git-lfs.github.com/spec/v1"

def find_repo_root():
    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (start / "evidence/mose-task1/review50/manifest.json").is_file():
            return start
    raise FileNotFoundError("Run this notebook from inside the paper-reproductions checkout.")

REPO_ROOT = find_repo_root()
REVIEW_ROOT = REPO_ROOT / "evidence/mose-task1/review50"

def artifact_data_url(source, mime_type):
    if not source.is_file():
        raise FileNotFoundError(source)
    payload = source.read_bytes()
    if payload.startswith(LFS_POINTER_PREFIX):
        raise RuntimeError(
            f"{source} is a Git LFS pointer, not the real media file. "
            "From the repository root run: python3 scripts/prepare_mose_notebooks.py"
        )
    return f"data:{mime_type};base64,{base64.b64encode(payload).decode('ascii')}"

def show_videos(items, columns=3):
    cards = []
    for title, relative, explanation in items:
        if relative is None:
            cards.append(
                f"<section class='media-card'><h4>{html.escape(title)}</h4>"
                f"<p><strong>Not applicable.</strong></p><p>{html.escape(explanation)}</p></section>"
            )
            continue
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "video/mp4")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<video controls muted loop playsinline preload='metadata' src='{data_url}'></video>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this video</a></p>"
            "</section>"
        )
    display(HTML(
        f"<style>.media-grid{{display:grid;grid-template-columns:repeat({columns},minmax(220px,1fr));gap:16px}}"
        ".media-card{border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#fff;color:#24292f}"
        ".media-card video{width:100%;background:#000}.media-card h4{margin:0 0 8px}.media-card p{font-size:13px}</style>"
        f"<div class='media-grid'>{''.join(cards)}</div>"
    ))

def show_images(items, columns=2):
    cards = []
    for title, relative, explanation in items:
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "image/png")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<img style='width:100%' src='{data_url}' alt='{html.escape(title)}'>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this image</a></p>"
            "</section>"
        )
    display(HTML(
        f"<div class='media-grid' style='grid-template-columns:repeat({columns},minmax(260px,1fr))'>"
        f"{''.join(cards)}</div>"
    ))

def show_metadata(paths):
    for relative in paths:
        path = REVIEW_ROOT / relative
        payload = json.loads(path.read_text())
        display(Markdown(f"#### `{relative}`"))
        print(json.dumps(payload, indent=2))

print(f"Loaded evidence from {REVIEW_ROOT}")


## 1. Sample `02b72b8d`


### Source and unwatermarked removal reference

The source has 67 frames at 960 x 480, 25 FPS, and a duration of about 2.68 seconds. The raw mask marks the object selected for removal. The three unwatermarked outputs show how ProPainter, DiffuEraser, and EffectErase behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/02b72b8d/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/02b72b8d/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/02b72b8d/diffueraser.mp4) | [EffectErase reference](../../evidence/mose-task1/review50/media/02b72b8d/baseline/effecterase.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/02b72b8d/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/02b72b8d/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/02b72b8d/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/02b72b8d/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.'), ('EffectErase without watermark', 'media/02b72b8d/baseline/effecterase.mp4', 'Unwatermarked EffectErase removal reference.')])


### OmniGuard results

Every displayed method video retains the native 960 x 480 frames. Embedding took 27.592 s. The embedding PSNR is **49.2299 dB**, with MSE 0.7764 and MAE 0.6184.

| Metric | ProPainter | DiffuEraser | EffectErase |
| --- | ---: | ---: | ---: |
| Removal runtime | 82.442 s | 189.067 s | 330.388 s |
| Global pixel IoU | **0.0011** | **0.0015** | **0.0006** |
| Mean frame IoU | 0.0010 | 0.0013 | 0.0006 |
| Minimum frame IoU | 0.0000 | 0.0000 | 0.0000 |
| Maximum frame IoU | 0.0286 | 0.0341 | 0.0167 |
| Intersection pixels | 973 | 1,402 | 423 |
| Union pixels | 871,773 | 924,514 | 694,556 |
| Message bit accuracy | N/A | N/A | N/A |

The localization record contains all three attacked videos. Its latest detector pass took 38.564 s. The global IoU compares the displayed binary prediction with the effective mask actually used by each remover.


In [ ]:
show_videos([('OmniGuard input', 'media/02b72b8d/original.mp4', 'Original MOSE source at native 960 x 480 resolution.'), ('OmniGuard aligned mask', 'media/02b72b8d/gt_mask.mp4', 'Native selected-object mask before remover preprocessing.'), ('OmniGuard watermarked', 'media/02b72b8d/methods/omniguard/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/02b72b8d/methods/omniguard/propainter.mp4', 'ProPainter output from the OmniGuard-watermarked input.'), ('ProPainter effective GT', 'media/02b72b8d/methods/omniguard/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing; the IoU reference.'), ('ProPainter predicted mask', 'media/02b72b8d/methods/omniguard/pred-mask-propainter.mp4', 'Native OmniGuard localization after ProPainter.'), ('DiffuEraser attacked video', 'media/02b72b8d/methods/omniguard/diffueraser.mp4', 'DiffuEraser output from the OmniGuard-watermarked input.'), ('DiffuEraser effective GT', 'media/02b72b8d/methods/omniguard/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing; the IoU reference.'), ('DiffuEraser predicted mask', 'media/02b72b8d/methods/omniguard/pred-mask-diffueraser.mp4', 'Native OmniGuard localization after DiffuEraser.'), ('EffectErase attacked video', 'media/02b72b8d/methods/omniguard/effecterase.mp4', 'EffectErase output from the OmniGuard-watermarked input.'), ('EffectErase effective GT', 'media/02b72b8d/methods/omniguard/effective-gt-mask-effecterase.mp4', 'Mask after EffectErase bilinear preprocessing; the IoU reference.'), ('EffectErase predicted mask', 'media/02b72b8d/methods/omniguard/pred-mask-effecterase.mp4', 'Native OmniGuard localization after EffectErase.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records, including checkpoints, repository commits, thresholds, runtimes, per-frame values, and file fingerprints.


In [ ]:
show_metadata(['media/02b72b8d/methods/omniguard/embed.json', 'media/02b72b8d/methods/omniguard/propainter.json', 'media/02b72b8d/methods/omniguard/diffueraser.json', 'media/02b72b8d/methods/omniguard/effecterase.json', 'media/02b72b8d/methods/omniguard/localization.json'])


## 2. Sample `0ab75563`


### Source and unwatermarked removal reference

The source has 104 frames at 960 x 448, 25 FPS, and a duration of about 4.16 seconds. The raw mask marks the object selected for removal. The three unwatermarked outputs show how ProPainter, DiffuEraser, and EffectErase behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4) | [EffectErase reference](../../evidence/mose-task1/review50/media/0ab75563/baseline/effecterase.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0ab75563/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0ab75563/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0ab75563/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0ab75563/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.'), ('EffectErase without watermark', 'media/0ab75563/baseline/effecterase.mp4', 'Unwatermarked EffectErase removal reference.')])


### OmniGuard results

Every displayed method video retains the native 960 x 480 frames. Embedding took 39.995 s. The embedding PSNR is **50.6420 dB**, with MSE 0.5609 and MAE 0.5280.

| Metric | ProPainter | DiffuEraser | EffectErase |
| --- | ---: | ---: | ---: |
| Removal runtime | 117.300 s | 244.393 s | 362.880 s |
| Global pixel IoU | **0.0007** | **0.0006** | **0.0004** |
| Mean frame IoU | 0.3943 | 0.3655 | 0.4039 |
| Minimum frame IoU | 0.0000 | 0.0000 | 0.0000 |
| Maximum frame IoU | 1.0000 | 1.0000 | 1.0000 |
| Intersection pixels | 347 | 323 | 159 |
| Union pixels | 500,143 | 513,975 | 420,087 |
| Message bit accuracy | N/A | N/A | N/A |

The localization record contains all three attacked videos. Its latest detector pass took 57.783 s. The global IoU compares the displayed binary prediction with the effective mask actually used by each remover.


In [ ]:
show_videos([('OmniGuard input', 'media/0ab75563/original.mp4', 'Original MOSE source at native 960 x 480 resolution.'), ('OmniGuard aligned mask', 'media/0ab75563/gt_mask.mp4', 'Native selected-object mask before remover preprocessing.'), ('OmniGuard watermarked', 'media/0ab75563/methods/omniguard/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/0ab75563/methods/omniguard/propainter.mp4', 'ProPainter output from the OmniGuard-watermarked input.'), ('ProPainter effective GT', 'media/0ab75563/methods/omniguard/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing; the IoU reference.'), ('ProPainter predicted mask', 'media/0ab75563/methods/omniguard/pred-mask-propainter.mp4', 'Native OmniGuard localization after ProPainter.'), ('DiffuEraser attacked video', 'media/0ab75563/methods/omniguard/diffueraser.mp4', 'DiffuEraser output from the OmniGuard-watermarked input.'), ('DiffuEraser effective GT', 'media/0ab75563/methods/omniguard/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing; the IoU reference.'), ('DiffuEraser predicted mask', 'media/0ab75563/methods/omniguard/pred-mask-diffueraser.mp4', 'Native OmniGuard localization after DiffuEraser.'), ('EffectErase attacked video', 'media/0ab75563/methods/omniguard/effecterase.mp4', 'EffectErase output from the OmniGuard-watermarked input.'), ('EffectErase effective GT', 'media/0ab75563/methods/omniguard/effective-gt-mask-effecterase.mp4', 'Mask after EffectErase bilinear preprocessing; the IoU reference.'), ('EffectErase predicted mask', 'media/0ab75563/methods/omniguard/pred-mask-effecterase.mp4', 'Native OmniGuard localization after EffectErase.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records, including checkpoints, repository commits, thresholds, runtimes, per-frame values, and file fingerprints.


In [ ]:
show_metadata(['media/0ab75563/methods/omniguard/embed.json', 'media/0ab75563/methods/omniguard/propainter.json', 'media/0ab75563/methods/omniguard/diffueraser.json', 'media/0ab75563/methods/omniguard/effecterase.json', 'media/0ab75563/methods/omniguard/localization.json'])


## 3. Sample `0b02886a`


### Source and unwatermarked removal reference

The source has 30 frames at 960 x 640, 25 FPS, and a duration of about 1.20 seconds. The raw mask marks the object selected for removal. The three unwatermarked outputs show how ProPainter, DiffuEraser, and EffectErase behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0b02886a/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0b02886a/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0b02886a/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0b02886a/diffueraser.mp4) | [EffectErase reference](../../evidence/mose-task1/review50/media/0b02886a/baseline/effecterase.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0b02886a/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0b02886a/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0b02886a/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0b02886a/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.'), ('EffectErase without watermark', 'media/0b02886a/baseline/effecterase.mp4', 'Unwatermarked EffectErase removal reference.')])


### OmniGuard results

Every displayed method video retains the native 960 x 480 frames. Embedding took 16.021 s. The embedding PSNR is **50.8190 dB**, with MSE 0.5385 and MAE 0.5059.

| Metric | ProPainter | DiffuEraser | EffectErase |
| --- | ---: | ---: | ---: |
| Removal runtime | 50.669 s | 105.697 s | 190.576 s |
| Global pixel IoU | **0.0054** | **0.0001** | **0.0038** |
| Mean frame IoU | 0.0038 | 0.0001 | 0.0039 |
| Minimum frame IoU | 0.0000 | 0.0000 | 0.0000 |
| Maximum frame IoU | 0.0365 | 0.0013 | 0.0349 |
| Intersection pixels | 9,310 | 218 | 5,512 |
| Union pixels | 1,732,155 | 1,768,264 | 1,450,886 |
| Message bit accuracy | N/A | N/A | N/A |

The localization record contains all three attacked videos. Its latest detector pass took 19.657 s. The global IoU compares the displayed binary prediction with the effective mask actually used by each remover.


In [ ]:
show_videos([('OmniGuard input', 'media/0b02886a/original.mp4', 'Original MOSE source at native 960 x 480 resolution.'), ('OmniGuard aligned mask', 'media/0b02886a/gt_mask.mp4', 'Native selected-object mask before remover preprocessing.'), ('OmniGuard watermarked', 'media/0b02886a/methods/omniguard/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/0b02886a/methods/omniguard/propainter.mp4', 'ProPainter output from the OmniGuard-watermarked input.'), ('ProPainter effective GT', 'media/0b02886a/methods/omniguard/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing; the IoU reference.'), ('ProPainter predicted mask', 'media/0b02886a/methods/omniguard/pred-mask-propainter.mp4', 'Native OmniGuard localization after ProPainter.'), ('DiffuEraser attacked video', 'media/0b02886a/methods/omniguard/diffueraser.mp4', 'DiffuEraser output from the OmniGuard-watermarked input.'), ('DiffuEraser effective GT', 'media/0b02886a/methods/omniguard/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing; the IoU reference.'), ('DiffuEraser predicted mask', 'media/0b02886a/methods/omniguard/pred-mask-diffueraser.mp4', 'Native OmniGuard localization after DiffuEraser.'), ('EffectErase attacked video', 'media/0b02886a/methods/omniguard/effecterase.mp4', 'EffectErase output from the OmniGuard-watermarked input.'), ('EffectErase effective GT', 'media/0b02886a/methods/omniguard/effective-gt-mask-effecterase.mp4', 'Mask after EffectErase bilinear preprocessing; the IoU reference.'), ('EffectErase predicted mask', 'media/0b02886a/methods/omniguard/pred-mask-effecterase.mp4', 'Native OmniGuard localization after EffectErase.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records, including checkpoints, repository commits, thresholds, runtimes, per-frame values, and file fingerprints.


In [ ]:
show_metadata(['media/0b02886a/methods/omniguard/embed.json', 'media/0b02886a/methods/omniguard/propainter.json', 'media/0b02886a/methods/omniguard/diffueraser.json', 'media/0b02886a/methods/omniguard/effecterase.json', 'media/0b02886a/methods/omniguard/localization.json'])


## Validation and provenance

- The full-review validator decoded all 50 OmniGuard result sets and checked frame counts, dimensions, FPS, mask foreground, and displayed-mask IoU against the JSON result.
- Full 50-sample validation report: [validation.json](../../evidence/mose-task1/review50/validation.json)
- Three-example notebook validation: [image-method-notebook-validation.json](../../evidence/mose-task1/review50/image-method-notebook-validation.json)
- Full 50-sample aggregate metrics: [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json)
- Experimental contract: [PROTOCOL.md](../../evidence/mose-task1/PROTOCOL.md)
- Official repository: [OmniGuard](https://github.com/xuanyuzhang21/OmniGuard)
- DiffuEraser implementation used for every removal result: [lixiaowen-xw/DiffuEraser](https://github.com/lixiaowen-xw/DiffuEraser)
- EffectErase implementation used for every removal result: [FudanCVL/EffectErase](https://github.com/FudanCVL/EffectErase)

The displayed values come from the committed OmniGuard evidence records.


## Recorded one-clip four-inpainter completion

All four removers use the same frozen watermarked input and aligned binary mask for this sample. Method-specific resizing, padding, and mask dilation are recorded in the adjacent JSON rather than silently normalized.

| Inpainting method | Status | Evidence |
|---|---:|---|
| ProPainter | complete | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/propainter.mp4) | [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/propainter.json) |
| DiffuEraser | complete | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/diffueraser.mp4) | [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/diffueraser.json) |
| EffectErase | complete | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/effecterase.mp4) | [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/effecterase.json) |
| E2FGVI-HQ | complete | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/e2fgvi.mp4) | [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/omniguard/e2fgvi.json) |

Watermark method: **Omniguard**.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'evidence/mose-task1/review50/media/02b72b8d/methods/omniguard'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4", "effecterase.mp4", "e2fgvi.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))
for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(JSON(json.loads(path.read_text())))
